# AI 3D Studio — verified GPU render gate

This is a prepared notebook, **not evidence of a completed Kaggle render**.
Attach the private dataset containing `render-bundle.zip`, enable Internet and a GPU,
then **Save Version → Save & Run All** to retain `/kaggle/working` output.
No production render starts unless a real Cycles GPU smoke render succeeds.
This runs your trusted Blender/Python project code. Inspect third-party assets/scripts first.

The installer targets Blender **5.2.3** to match the cloud headless runtime.
The exact standalone release URL and Kaggle installation have not been validated yet;
the official SHA256 manifest must verify before any binary is extracted or executed.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, stat, subprocess, sys, zipfile

bundles = list(Path('/kaggle/input').rglob('render-bundle.zip'))
if len(bundles) != 1:
    raise RuntimeError(f'Attach exactly one studio dataset containing render-bundle.zip; found {len(bundles)}')
bundle = bundles[0]
workspace = Path('/kaggle/working/studio')
workspace.mkdir(exist_ok=True)
with bundle.open('rb') as stream:
    digest = hashlib.file_digest(stream, 'sha256').hexdigest()
marker = workspace / '.bundle-sha256'
if any(workspace.iterdir()) and (not marker.exists() or marker.read_text() != digest):
    raise RuntimeError('Different or partial bundle exists. Use a new notebook session to avoid overwriting files.')
with zipfile.ZipFile(bundle) as archive:
    members = archive.infolist()
    if len(members) > 100000 or sum(m.file_size for m in members) > 20 * 1024**3:
        raise RuntimeError('Bundle exceeds safe extraction limits (100000 files / 20 GiB)')
    for member in members:
        target = (workspace / member.filename).resolve()
        if not target.is_relative_to(workspace.resolve()) or stat.S_ISLNK(member.external_attr >> 16):
            raise RuntimeError('Unsafe bundle path/link: ' + member.filename)
    archive.extractall(workspace)
marker.write_text(digest)
for required in ['job.json', 'project/scene.blend', 'pipeline/__main__.py', 'kaggle/setup/install_blender.py']:
    if not (workspace / required).is_file():
        raise RuntimeError('Incomplete render bundle: ' + required)
# Check every recorded payload before executing trusted studio code.
manifest = json.loads((workspace / 'manifest.json').read_text())
for relative, expected in manifest.items():
    path = (workspace / relative).resolve()
    if not path.is_relative_to(workspace.resolve()) or not path.is_file():
        raise RuntimeError('Invalid manifest entry: ' + relative)
    with path.open('rb') as stream:
        actual = hashlib.file_digest(stream, 'sha256').hexdigest()
    if actual != expected:
        raise RuntimeError('Bundle checksum mismatch: ' + relative)
print('Verified payload files:', len(manifest))
print('Bundle SHA256:', digest)
print('Job:', json.loads((workspace / 'job.json').read_text()))


In [ ]:
# Blender is installed on Kaggle only. Official SHA256 verification remains enabled.
sys.path.insert(0, str(workspace))
import importlib.util
spec = importlib.util.spec_from_file_location("studio_blender_install", workspace / "kaggle/setup/install_blender.py")
installer = importlib.util.module_from_spec(spec)
spec.loader.exec_module(installer)
install = installer.install
blender = install(Path('/kaggle/working/tools'))
subprocess.run([str(blender), '--version'], check=True)
if not shutil.which('ffmpeg'):
    raise RuntimeError('Kaggle image lacks ffmpeg: enable Internet and install ffmpeg through the image package manager.')
try:
    subprocess.run(['nvidia-smi'], check=True)
except (FileNotFoundError, subprocess.CalledProcessError) as exc:
    raise RuntimeError('No usable NVIDIA GPU. Select an available GPU in Notebook settings; no CPU fallback.') from exc


In [ ]:
# This is an actual tiny Cycles GPU render, not just device enumeration.
gpu_evidence = Path('/kaggle/working/gpu-evidence.json')
subprocess.run([str(blender), '--background', '--factory-startup', '--python-exit-code', '1',
                '--python', str(workspace / 'kaggle/gpu_test.py'), '--', '--out', str(gpu_evidence)],
               cwd=workspace, check=True)
evidence = json.loads(gpu_evidence.read_text())
if not evidence.get('actual_gpu_render_passed'):
    raise RuntimeError('GPU smoke render was not confirmed')
print(json.dumps(evidence, indent=2))
# Use the backend that actually rendered; an OptiX failure must not recur under AUTO.
validated_job = Path('/kaggle/working/validated-job.json')
job = json.loads((workspace / 'job.json').read_text())
if job.get('device') != 'GPU' or job.get('engine') != 'CYCLES':
    raise RuntimeError('Kaggle notebook requires an explicit Cycles GPU job')
job['backend'] = evidence['configuration']['backend']
validated_job.write_text(json.dumps(job, indent=2))


In [ ]:
# Logs, progress, checkpoints and frames survive only if Kaggle retains the version output.
output = Path('/kaggle/working/render')
log = Path('/kaggle/working/render.log')
command = [sys.executable, '-m', 'pipeline', '--blender', str(blender), 'render',
           '--job', str(validated_job), '--blend', 'project/scene.blend', '--out', str(output)]
with log.open('a') as stream:
    process = subprocess.Popen(command, cwd=workspace, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end='')
        stream.write(line)
        stream.flush()
    returncode = process.wait()
if returncode:
    raise RuntimeError(f'Render failed with exit {returncode}; inspect render.log and progress files')


In [ ]:
job = json.loads((workspace / 'job.json').read_text())
movie = Path('/kaggle/working/animation.mp4')
subprocess.run([sys.executable, '-m', 'pipeline', 'encode', '--frames', str(output / 'frames'),
                '--fps', str(job['fps']), '--out', str(movie)], cwd=workspace, check=True)
# A single archive helps retrieve large image sequences without output-list pagination surprises.
archive = shutil.make_archive('/kaggle/working/render-artifacts', 'zip', output)
print('Encoded:', movie, 'bytes:', movie.stat().st_size)
print('Frames/checkpoints:', archive)
from IPython.display import Video, display
display(Video(str(movie), embed=True))


## Retrieve and recover

After **Save & Run All** completes, download `animation.mp4`, `gpu-evidence.json`, `render.log`,
and `render-artifacts.zip` from the notebook **Output** tab, or use the cloud CLI's
`kaggle kernels output owner/kernel/version -p artifacts/kaggle-run`.
Do not assume files from an interrupted session are retained. If a failed version has saved
partial frames, download them first and attach/reuse them in a fresh session; preserve its
job configuration and render-state metadata. Rerunning the render cell in the same session
uses the pipeline's validated frame checkpoints. Never bypass quota or session limits.
